# Part 5: Evaluating Chunking Strategies

This guide provides instructions for the next phase of optimising your RAG application: systematically testing different chunking strategies. Having established a baseline performance, we will now experiment with how we split our documents to see if we can improve retrieval quality.

> This notebook is a guide to enhancing your project. **Do not run the code cells here directly.** Follow the instructions in your local development environment.

---
## 1.&nbsp; The Importance of Chunking

The way we break down our documents into smaller pieces, or chunks, is one of the most critical factors in the performance of a RAG system.

* If chunks are too small, they may not contain enough context for the LLM to understand the information properly.
* If chunks are too large, they might contain too much irrelevant information, creating noise that confuses the retriever.

The goal is to find the "Goldilocks zone" for chunk size and overlap that provides the retriever with just the right amount of context.

---
## 2.&nbsp; Preparing for the Experiment

To test different chunking strategies, we'll add a new evaluation stage to our existing framework. This involves adding a new configuration list and a new function to our evaluation scripts.

### 2.1 Add the Chunking Strategy Configurations

First, we need to define the different chunk sizes and overlaps we want to test. We will centralise this in our evaluation configuration file.

1.  Open `evaluation/evaluation_config.py` in your VSCode editor.
2.  Add the following list to the end of the file. This list defines the experiments we will run.

In [ ]:
# --- Configuration for Chunking Strategy Evaluation ---
CHUNKING_STRATEGY_CONFIGS: list[dict[str, int]] = [
    {'size': 512, 'overlap': 50},
    {'size': 768, 'overlap': 115},
    {'size': 1024, 'overlap': 200},
]

This setup makes it easy to add more experimental configurations in the future without altering the evaluation logic.

### 2.3 Create the Chunking Evaluation Function

Now, we'll add a new function to `evaluation/evaluation_engine.py` that is responsible for testing our chunking strategies.

1.  Open `evaluation/evaluation_engine.py` in your VSCode editor.
2.  First, import the new configuration list by importing `CHUNKING_STRATEGY_CONFIGS` from `evaluation.evaluation_config`.

In [ ]:
from evaluation.evaluation_config import (
    CHUNKING_STRATEGY_CONFIGS,
)

3. Now, add the following function to the file, placing it after the `evaluate_baseline` function.

In [ ]:
def evaluate_chunking_strategies() -> None:
    """ Evaluates different chunk sizes and overlaps. """
    print("\n--- 🚀 Stage 2: Evaluating Chunking Strategies ---")

    llm_to_test: Groq = initialise_llm()

    embed_model_to_test: HuggingFaceEmbedding = get_embedding_model()

    questions, ground_truths = get_evaluation_data()

    ragas_llm: LlamaIndexLLMWrapper
    ragas_embeddings: HuggingFaceEmbeddings
    ragas_llm, ragas_embeddings = load_ragas_models()

    all_results: list[pd.DataFrame] = []

    for config in CHUNKING_STRATEGY_CONFIGS:

        chunk_size, chunk_overlap = config['size'], config['overlap']

        print(f"--- Testing Chunk Config: size={chunk_size}, "
              f"overlap={chunk_overlap} ---")

        index: VectorStoreIndex = get_or_build_index(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            embed_model=embed_model_to_test
        )

        query_engine: BaseQueryEngine = index.as_query_engine(
            similarity_top_k=SIMILARITY_TOP_K,
            llm=llm_to_test
        )

        qa_dataset: Dataset = generate_qa_dataset(
            query_engine,
            questions,
            ground_truths
        )

        print("--- Running Ragas evaluation for chunking... ---")

        # --- If you don't have a Rate per Minute limit on your API ---
        # results_df: pd.DataFrame = evaluate_without_rate_limit(
        #     qa_dataset,
        #     ragas_llm,
        #     ragas_embeddings,
        # )

        # --- If you do have a Rate per Minute API limit ---
        results_df: pd.DataFrame = evaluate_with_rate_limit(
            qa_dataset,
            ragas_llm,
            ragas_embeddings,
        )

        # Add Chunk Size and Chunk Overlap to DataFrame to help tracking
        results_df['chunk_size'] = chunk_size
        results_df['chunk_overlap'] = chunk_overlap

        all_results.append(results_df)

    final_df: pd.DataFrame = pd.concat(all_results, ignore_index=True)

    save_results(final_df, "chunking_evaluation")

    print("--- ✅ Chunking Strategy Evaluation Complete ---")

This function iterates through each configuration in `CHUNKING_STRATEGY_CONFIGS`, builds a unique vector store for that configuration, runs the Ragas evaluation, and then combines all the results into a single DataFrame.

### 2.4 Update the Main Execution Block

Finally, we need to call our new function.

In `evaluate.py`, update the `if __name__ == "__main__":` block at the end of the file to include a call to `evaluate_chunking_strategies`.

> Remember to comment out most of the questions to save your API calls while you're testing the code.

In [ ]:
from evaluation.evaluation_engine import (
    # evaluate_baseline,
    evaluate_chunking_strategies,
)


if __name__ == "__main__":

    # Run Stage 1: Baseline Evaluation
    # evaluate_baseline()

    # Run Stage 2: Chunking Strategy Evaluation
    evaluate_chunking_strategies()

---
## 3.&nbsp; Run the Chunking Evaluation

With the new function and configurations in place, you're ready to run the experiment.

From your VSCode terminal, run the evaluation script as before:

In [ ]:
python evaluate.py

The script will first run the baseline evaluation and then proceed to Stage 2, testing each chunking strategy in turn. It will create and save a separate vector store for each strategy in the `local_storage/experimental_vector_stores` directory.

---
## 4.&nbsp; Analyse the Results

Once the script finishes, navigate to the `evaluation/evaluation_results` folder. You will find two new files:

1.  **`chunking_evaluation_detailed_... .csv`**: Contains the row-by-row scores for every question under each chunking strategy.
2.  **`chunking_evaluation_summary_... .csv`**: This is the most important file. It shows the average scores for each metric, grouped by `chunk_size` and `chunk_overlap`.

Open the summary file. Your goal is to find the combination of `chunk_size` and `chunk_overlap` that yields the best overall scores, particularly for **`context_recall`** and **`context_precision`**, as these are most directly affected by the quality of the retrieved chunks.

---
## 5.&nbsp; Challenges and Further Exploration 😀

You've now established a repeatable process for optimising a crucial component of your RAG pipeline. The next step is to apply these techniques to the custom RAG chatbot you have been building with your own data.

### 1. Find Your Optimal Strategy for Your Project

Now it's time to run the chunking evaluation on your own RAG system and analyse the results to find the best settings for your specific documents.

**Your Mission:**

1.  **Run the Full Evaluation:**

2.  **Set Up Your Analysis Notebook:**

      * In your `notebooks` directory, create a new Notebook file named `02_chunking_analysis.ipynb`.

3.  **Load and Analyse Your Results:**

      * In the new notebook, load the summary results from your chunking experiment.
      * Analyse the trade-offs. Which combination of `chunk_size` and `chunk_overlap` gives the best `context_recall` and `context_precision`? Create plots to help visualise the differences.

4.  **Document Your Decision and Update Your Application:**

      * In a markdown cell, write a conclusion explaining which chunking strategy you have chosen as the new optimum for your project and justify your decision using the data.
      * Open `src/config.py` and update the `CHUNK_SIZE` and `CHUNK_OVERLAP` variables with your new, optimised values. Your main chatbot will now use this improved configuration.

### 2. Expand the Experiment

The three strategies we tested are just a starting point. Now that you have a robust evaluation pipeline, you can easily conduct more advanced experiments.

**Challenge:**

  * Return to `evaluation/evaluation_config.py` and add more configurations to the `CHUNKING_STRATEGY_CONFIGS` list. Try a wider range of sizes and overlaps.
  * Does a very large chunk size (e.g., 2048) improve performance by providing more context, or does it introduce too much noise and lower the scores?
  * What happens with a very small chunk size (e.g., 256)? Does it improve precision at the cost of recall?
  * Run the evaluation again and analyse the new results to see if you can improve performance further.

  ### 3. Update your Config file
  Once you've found a chunking strategy that you're happy with, make sure you update the `src/config` file with the best combination of chunk and overlap.